In [2]:
import duckdb
import pandas as pd
from pathlib import Path

# Rutas del proyecto
RUTA_PROYECTO = Path("..").resolve()
RUTA_DATA_RAW = RUTA_PROYECTO / "data" / "raw"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

print(f"Proyecto: {RUTA_PROYECTO}")
print(f"Datos crudos: {RUTA_DATA_RAW}")
print(f"Base DuckDB: {RUTA_DUCKDB}")
print(f"\nArchivos en raw:")
for f in sorted(RUTA_DATA_RAW.iterdir()):
    print(f"  {f.name}")

Proyecto: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark
Datos crudos: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\data\raw
Base DuckDB: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb

Archivos en raw:
  .ipynb_checkpoints
  dim_cliente.csv
  dim_cliente_old.csv
  fact_lineas_pedido.csv
  fact_lineas_pedido_old.csv
  mosaic.csv
  ventas_minoristas.csv
  ventas_minoristas_2022_old.csv
  ventas_minoristas_2023_old.csv
  ventas_minoristas_2024_old.csv
  ventas_minoristas_2025_old.csv


In [3]:
# Conectar a DuckDB (crea el archivo si no existe)
con = duckdb.connect(str(RUTA_DUCKDB))

# Crear los 3 esquemas
con.execute("CREATE SCHEMA IF NOT EXISTS bronze")
con.execute("CREATE SCHEMA IF NOT EXISTS silver")
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

# Verificar
esquemas = con.execute("SELECT schema_name FROM information_schema.schemata ORDER BY schema_name").fetchdf()
print("Esquemas disponibles:")
print(esquemas)

Esquemas disponibles:
          schema_name
0              bronze
1                gold
2  information_schema
3                main
4                main
5                main
6          pg_catalog
7              silver


In [4]:
# Cargar dim_cliente
con.execute(f"""
    CREATE OR REPLACE TABLE bronze.dim_cliente AS
    SELECT * FROM read_csv_auto('{RUTA_DATA_RAW}/dim_cliente.csv',
                                 header=true,
                                 sample_size=-1,
                                 all_varchar=true)
""")
print("bronze.dim_cliente cargada")

# Cargar fact_lineas_pedido
con.execute(f"""
    CREATE OR REPLACE TABLE bronze.fact_lineas_pedido AS
    SELECT * FROM read_csv_auto('{RUTA_DATA_RAW}/fact_lineas_pedido.csv',
                                 header=true,
                                 sample_size=-1)
""")
print("bronze.fact_lineas_pedido cargada")

# Cargar mosaic
con.execute(f"""
    CREATE OR REPLACE TABLE bronze.mosaic AS
    SELECT * FROM read_csv_auto('{RUTA_DATA_RAW}/mosaic.csv',
                                 header=true,
                                 sample_size=-1,
                                 all_varchar=true)
""")
print("bronze.mosaic cargada")

bronze.dim_cliente cargada
bronze.fact_lineas_pedido cargada
bronze.mosaic cargada


In [5]:
# Cargar ventas_minoristas desde el archivo único (contiene 2019-2026, sin filtrar)
con.execute(f"""
    CREATE OR REPLACE TABLE bronze.ventas_minoristas AS
    SELECT * FROM read_csv_auto('{RUTA_DATA_RAW}/ventas_minoristas.csv',
                                 header=true,
                                 sample_size=-1)
""")
print("bronze.ventas_minoristas cargada (archivo único 2019-2026, sin filtrar)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

bronze.ventas_minoristas cargada (archivo único 2019-2026, sin filtrar)


In [6]:
# Verificar que todo se ha cargado correctamente
print("RESUMEN DE TABLAS BRONZE\n")
print(f"{'Tabla':<35} {'Filas':>15}")
print("-" * 55)

tablas = ['dim_cliente', 'fact_lineas_pedido', 'mosaic', 'ventas_minoristas']
for tabla in tablas:
    n = con.execute(f"SELECT COUNT(*) FROM bronze.{tabla}").fetchone()[0]
    print(f"bronze.{tabla:<28} {n:>15,}")

# Tamaño de la base
tamanio_mb = RUTA_DUCKDB.stat().st_size / (1024 * 1024)
print(f"\nTamaño de selmark.duckdb: {tamanio_mb:.2f} MB")

RESUMEN DE TABLAS BRONZE

Tabla                                         Filas
-------------------------------------------------------
bronze.dim_cliente                            3,492
bronze.fact_lineas_pedido                   470,963
bronze.mosaic                                 6,457
bronze.ventas_minoristas                  4,009,222

Tamaño de selmark.duckdb: 306.76 MB


In [7]:
con.close()
print("Conexión cerrada. Base de datos guardada en disco.")

Conexión cerrada. Base de datos guardada en disco.
